# Restricted-input method reference
Source: analysis/intra-park/road_connectivity.ipynb
Requires independently obtained inputs and source directory layout. Not an end-to-end pipeline. Outputs can contain sensitive data; keep the working directory private. External absolute paths have been replaced by external_inputs/<basename>. See docs/REPRODUCIBILITY.md.


In [ ]:
import os
from pathlib import Path
private_root = Path(os.environ["UFUG_PRIVATE_ROOT"]).resolve()
os.chdir(private_root)


In [ ]:
# Road Network Connectivity.ipynb
# You can paste this script into a Jupyter Notebook cell.

# --- 1. Imports ---
import geopandas as gpd
import numpy as np
import networkx as nx
from shapely.geometry import LineString
from shapely.ops import unary_union
from sklearn.cluster import DBSCAN
from dataclasses import dataclass
from typing import Dict, Any, Literal, Optional

import os
# Working directory is explicitly configured in the first cell.


In [ ]:
@dataclass
class ConnectivityResult:
    one_value: float
    one_value_name: str
    metrics: Dict[str, Any]


def explode_lines(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    gdf = gdf.explode(index_parts=False, ignore_index=True)
    gdf = gdf[gdf.geometry.type == "LineString"].copy()
    gdf.reset_index(drop=True, inplace=True)
    return gdf

def line_endpoints_xy(lines: gpd.GeoDataFrame) -> np.ndarray:
    """Extract start and end coordinates from every LineString."""
    pts = []
    for geom in lines.geometry.values:
        if isinstance(geom, LineString):
            x0, y0 = geom.coords[0]
            x1, y1 = geom.coords[-1]
            pts.append((x0, y0))
            pts.append((x1, y1))
    return np.asarray(pts, dtype=float)


def segment_endpoints(lines: gpd.GeoDataFrame) -> np.ndarray:
    pts = []
    for geom in lines.geometry.values:
        if isinstance(geom, LineString):
            x0, y0 = geom.coords[0]
            x1, y1 = geom.coords[-1]
            pts.append((x0, y0))
            pts.append((x1, y1))
    return np.asarray(pts, dtype=float)


def cluster_endpoints(coords_xy: np.ndarray, snap_tolerance: float) -> np.ndarray:
    if len(coords_xy) == 0:
        return np.array([], dtype=int)
    labels = DBSCAN(eps=snap_tolerance, min_samples=1).fit(coords_xy).labels_
    return labels


def build_graph(lines: gpd.GeoDataFrame, labels: np.ndarray, coords_xy: np.ndarray) -> nx.Graph:
    G = nx.Graph()
    clusters = {}
    for (x, y), lab in zip(coords_xy, labels):
        clusters.setdefault(lab, []).append((x, y))

    for lab, points in clusters.items():
        arr = np.asarray(points)
        cx, cy = arr.mean(axis=0)
        G.add_node(int(lab), x=float(cx), y=float(cy))

    idx = 0
    for geom in lines.geometry.values:
        x0, y0 = geom.coords[0]
        x1, y1 = geom.coords[-1]
        lab0 = labels[idx]
        lab1 = labels[idx + 1]
        idx += 2

        if lab0 == lab1:
            continue
        length_m = float(geom.length)
        u, v = int(lab0), int(lab1)
        if G.has_edge(u, v):
            if length_m < G[u][v]["weight"]:
                G[u][v]["weight"] = length_m
        else:
            G.add_edge(u, v, weight=length_m)
    return G


def compute_metrics(G: nx.Graph) -> Dict[str, Any]:
    n = G.number_of_nodes()
    m = G.number_of_edges()
    comps = list(nx.connected_components(G))
    c = len(comps)
    n_lcc = max((len(s) for s in comps), default=0)
    lcc_ratio = (n_lcc / n) if n > 0 else 0.0
    density = nx.density(G) if n > 1 else 0.0
    avg_deg = (2 * m / n) if n > 0 else 0.0
    try:
        avg_clust = nx.average_clustering(G)
    except Exception:
        avg_clust = None

    metrics = {
        "nodes": n,
        "edges": m,
        "components": c,
        "largest_component_nodes": n_lcc,
        "giant_component_ratio": lcc_ratio,
        "edge_density": density,
        "average_degree": avg_deg,
        "average_clustering": avg_clust,
    }
    return metrics


def connectivity_from_geojson(
    path_to_geojson: str,
    snap_tolerance_m: float = 5.0,
    one_value: Literal[
        "giant_component_ratio",
        "edge_density",
        "average_clustering",
        "average_degree",
    ] = "giant_component_ratio",
) -> ConnectivityResult:
    gdf = gpd.read_file(path_to_geojson)
    lines = explode_lines(gdf)
    coords_xy = line_endpoints_xy(lines)

    if coords_xy.size == 0:
        return ConnectivityResult(one_value=float("nan"), one_value_name=one_value, metrics={})

    labels = cluster_endpoints(coords_xy, snap_tolerance=snap_tolerance_m)
    G = _build_graph(lines, labels, coords_xy)

    metrics = _compute_metrics(G)
    value = metrics.get(one_value, np.nan)
    return ConnectivityResult(one_value=value, one_value_name=one_value, metrics=metrics)


# --- Example Usage ---
# path = "./road_network.geojson"
# result = connectivity_from_geojson(path, snap_tolerance_m=5, one_value="giant_component_ratio")
# print(result)


# --- Option B: Node aggregation to hexes (fast & scalable) ---
# Compute node-level metrics once on the whole network, then spatially join nodes to hexes
# and aggregate (mean/median/sum). This yields a smooth per-hex surface of connectivity.

import pandas as pd
from shapely.geometry import Point


def build_city_graph(roads: gpd.GeoDataFrame, snap_tolerance_m: float = 5.0) -> nx.Graph:
    """Build an undirected, weighted graph for the *entire* road dataset."""
    lines = explode_lines(roads)
    coords = line_endpoints_xy(lines)
    if coords.size == 0:
        return nx.Graph()
    labels = cluster_endpoints(coords, snap_tolerance_m)
    G = build_graph(lines, labels, coords)
    return G


def nodes_gdf_from_graph(G: nx.Graph, crs) -> gpd.GeoDataFrame:
    rows = []
    for n, d in G.nodes(data=True):
        rows.append({"node": n, "geometry": Point(d["x"], d["y"])})
    return gpd.GeoDataFrame(rows, crs=crs)


def compute_node_metric(
    G: nx.Graph,
    metric: str = "degree",
    *,
    weight: str = "weight",
    betweenness_k: int | None = None,
    betweenness_seed: int = 42,
) -> dict[int, float]:
    """Compute a node-level metric on G.
    metric ∈ {"degree", "betweenness", "closeness"}.
    - degree: simple degree (valence)
    - betweenness: set k for faster approximation on large graphs
    - closeness: uses edge length as distance if present
    """
    if metric == "degree":
        return dict(G.degree())
    if metric == "betweenness":
        return nx.betweenness_centrality(G, normalized=True, weight=weight, k=betweenness_k, seed=betweenness_seed)
    if metric == "closeness":
        # For disconnected graphs, NetworkX computes closeness within component; acceptable for local analysis
        return nx.closeness_centrality(G, distance=weight)
    raise ValueError("Unsupported metric: choose 'degree', 'betweenness', or 'closeness'.")


def aggregate_node_metric_to_hex(
    G: nx.Graph,
    hexes: gpd.GeoDataFrame,
    *,
    metric: str = "degree",
    agg: str = "mean",
    weight: str = "weight",
    betweenness_k: int | None = None,
) -> gpd.GeoDataFrame:
    """Attach an aggregated node metric to each hexagon.
    - agg ∈ {"mean","median","sum"}
    - betweenness_k: sample size for approximate betweenness (None = exact; use e.g. 200–1000 for speed)
    """
    vals = compute_node_metric(G, metric=metric, weight=weight, betweenness_k=betweenness_k)

    nodes = nodes_gdf_from_graph(G, crs=hexes.crs)
    nodes["value"] = nodes["node"].map(vals).astype(float)

    joined = gpd.sjoin(nodes, hexes, predicate="within", how="left")
    grp = joined.groupby(joined.index_right)["value"]

    if agg == "mean":
        agg_vals = grp.mean()
    elif agg == "median":
        agg_vals = grp.median()
    elif agg == "sum":
        agg_vals = grp.sum()
    else:
        raise ValueError("agg must be 'mean', 'median', or 'sum'.")

    out = hexes.copy()
    colname = f"{metric}_{agg}"
    out[colname] = agg_vals.reindex(out.index).values
    return out

def aggregate_node_metric_to_hex_nearest(
    G: nx.Graph,
    hexes: gpd.GeoDataFrame,
    *,
    metric: str = "degree",
    agg: str = "mean",
    weight: str = "weight",
    betweenness_k: int | None = None,
    predicate: str = "within",
    roads: gpd.GeoDataFrame | None = None,
    road_buffer_m: float = 0.0,
    ) -> gpd.GeoDataFrame:
    """Aggregate node metric to hexes. If a hex has roads but no nodes,
    impute its value from the nearest node (to the hex centroid).
    """
    hexes = hexes.reset_index(drop=True)
    hexes["hex_id"] = hexes.index


    # Node metrics
    vals = compute_node_metric(G, metric=metric, weight=weight, betweenness_k=betweenness_k)
    nodes = nodes_gdf_from_graph(G, crs=hexes.crs).copy()
    nodes["value"] = nodes["node"].map(vals).astype(float)


    # Join nodes → hexes
    hex_min = hexes[["hex_id", "geometry"]].copy()
    joined = gpd.sjoin(nodes, hex_min, predicate=predicate, how="left")


    grp = joined.groupby("hex_id")["value"]
    if agg == "mean":
        agg_vals = grp.mean()
    elif agg == "median":
        agg_vals = grp.median()
    elif agg == "sum":
        agg_vals = grp.sum()
    else:
        raise ValueError("agg must be 'mean', 'median', or 'sum'.")


    out = hexes.copy()
    colname = f"{metric}_{agg}"
    out[colname] = out["hex_id"].map(agg_vals)

    # Impute for road-only hexes if roads provided
    if roads is not None and not nodes.empty:
        if road_buffer_m and road_buffer_m != 0:
            r = r.copy(); r["geometry"] = r.geometry.buffer(road_buffer_m)
        sidx = r.sindex
        has_road = []
        for geom in out.geometry.values:
            idx = list(sidx.query(geom, predicate="intersects"))
            has_road.append(bool(idx))
        out["_has_road"] = has_road


        # Hexes to impute
        mask = out[colname].isna() & out["_has_road"]
        if mask.any():
            hex_cent = out.loc[mask, ["hex_id", "geometry"]].copy()
            hex_cent["geometry"] = hex_cent.geometry.centroid
            try:
                nn = gpd.sjoin_nearest(hex_cent, nodes[["geometry", "value"]], how="left", distance_col="_dist")
                valmap = dict(zip(nn["hex_id"], nn["value"]))
                out.loc[mask, colname] = out.loc[mask, "hex_id"].map(valmap)
            except Exception:
                # fallback manual nearest search
                n_sidx = nodes.sindex
                fill_vals = {}
                for _, row in hex_cent.iterrows():
                    pt = row.geometry
                    cand = list(n_sidx.query(pt, predicate="intersects"))
                    if not cand:
                        cand = list(n_sidx.query(pt.buffer(50)))
                    if not cand:
                        continue
                    sub = nodes.iloc[cand]
                    d = sub.distance(pt)
                    j = int(d.idxmin())
                    fill_vals[row["hex_id"]] = float(sub.loc[j, "value"])
                if fill_vals:
                    out.loc[mask, colname] = out.loc[mask, "hex_id"].map(fill_vals)
        out = out.drop(columns=["_has_road"])

    return out.drop(columns=["hex_id"])


In [ ]:
path = "gis_files/road_network/pedestrian_accessible_road_network_merged.geojson"
gdf = gpd.read_file(path)
parks_boundaries = gpd.read_file('gis_files/parks_buffer_2km.geojson')

# intersect gdf with parks_boundaries with the attributes of parks_boundaries retained
gdf = gpd.overlay(gdf, parks_boundaries, how='intersection')
gdf['layer'].value_counts()

In [ ]:
for park in gdf['layer'].unique():
    print(f"Park: {park}")
    sub_gdf = gdf[gdf['layer'] == park]
    hex_path = f"gis_files/hexagons/{park}_hex_agg.geojson"
    hexes = gpd.read_file(hex_path)
    hexes.drop(columns=['index_right'], inplace=True)
    G = build_city_graph(sub_gdf, snap_tolerance_m=6)
    print(G.number_of_nodes(), "nodes", G.number_of_edges(), "edges")

    hex_cls = aggregate_node_metric_to_hex_nearest(G, hexes, metric="closeness", agg="mean")
    # hex_cls.to_file("hex_closeness_mean.gpkg", layer="hex", driver="GPKG")

    hex_cls = hex_cls[['closeness_mean', 'geometry', 'grid_id']]
    # plot 
    hex_cls.to_file(f"gis_files/hexagons/hex_closeness_mean/{park}_hex_closeness_mean.geojson", driver="GeoJSON")

In [ ]:
# -*- coding: utf-8 -*-
import os
from pathlib import Path
import matplotlib.pyplot as plt
import geopandas as gpd
import numpy as np

USE_CLIPPED_MIN_MAX = True  # ignore bottom/top 1% to avoid outliers

def get_vmin_vmax(s: np.ndarray):
    """Robust min/max for colour scaling."""
    s = s[~np.isnan(s)]
    if s.size == 0:
        return None, None
    if USE_CLIPPED_MIN_MAX:
        return (np.nanpercentile(s, 1), np.nanpercentile(s, 99))
    return (float(np.nanmin(s)), float(np.nanmax(s)))

def plot_perception_map(gdf: gpd.GeoDataFrame,
                        column: str,
                        title: str,
                        cmap: str,
                        out_path: Path):
    
    """Plot a single choropleth and save at 300 dpi."""
    # compute vmin/vmax
    vmin, vmax = get_vmin_vmax(gdf[column].to_numpy())
    if vmin is None or vmax is None or vmin == vmax:
        # fallback to simple values if empty or constant
        vmin, vmax = 0.0, 1.0

    fig, ax = plt.subplots(figsize=(8, 8))  # square works well for grids

    # draw polygons
    gdf.plot(
        column=column,
        ax=ax,
        cmap=cmap,
        linewidth=0.2,
        edgecolor="black",
        vmin=vmin,
        vmax=vmax,
        legend=True,
        legend_kwds={
            "label": title,
            "orientation": "vertical",
            "shrink": 0.5,        # half the colourbar height
            "fraction": 0.02      # thinner bar width (optional tweak)
        },
        missing_kwds={
            "color": "#f0f0f0",
            "edgecolor": "lightgrey",
            "hatch": "///",
            "label": "No data"
        }
    )

    ax.set_title(title, fontsize=14)
    ax.set_axis_off()
    ax.set_aspect('equal')  # grids are in a projected CRS → preserve shape

    # ensure folder exists and save
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

parks = ["amk", "jurong", "punggol"] 
for park in parks:
    print(f"Processing {park}")
    in_fp = f"gis_files/hexagons/hex_closeness_mean/{park}_hex_closeness_mean.geojson"
    gdf = gpd.read_file(in_fp)
    attri_to_plot = "closeness_mean"
    cmap = "viridis"
    out_fp = f"{park}_{attri_to_plot}.png"
    plot_title = None

    plot_perception_map(
        gdf=gdf,
        column=attri_to_plot,
        title=plot_title,
        cmap=cmap,
        out_path=out_fp
    )

print("Done.")